> **STAGE-2 SUPERSEDED NOTICE:** This Stage-1 notebook is superseded by **[`10_validated_semantic_diversity.ipynb`](file://./notebooks/10_validated_semantic_diversity.ipynb)**, which audits the `seed0`-only provenance of `human_eval_blinded_sheet.csv`, separates Task-1 and Task-2 truncation lengths, adds LSA $k$-sensitivity, PSD Vendi Score bootstrap CIs, Holm–Bonferroni correction, and `SentenceTransformer` multi-seed evaluation.

# Notebook 05 — Phase 5: Semantic Diversity, Representation Anisotropy, & Metric Validation on 600 Real Passages

**Purpose:**
1. Evaluate complementary contextual and topical semantic diversity metrics on the **600 real generated passages** (`M1_ELFM`, `S1_ELFB`, `Ref1_ELFB`) stored on disk:
   - **Pretrained `t5-small` Encoder (`512`-d):** Raw mean-pooled token states and **All-but-the-Top isotropic** representations (Mu & Viswanath, 2018).
   - **Pretrained `gpt2` (`768`-d):** Raw last-layer hidden states (demonstrating the severe Ethayarajh 2019 anisotropy cone where raw cosine distance is $\sim 0.0026$ and Vendi Score $\approx 1.02$) vs. **All-but-the-Top isotropic** representations.
   - **Content-Word TF-IDF + 32-dim SVD (`LSA` Topic Space):** Measures thematic/topical dispersion across passages independently of function-word syntax.
   - **Vendi Score (Friedman & Dieng, 2023):** Effective number of unique semantic modes $\exp(-\sum_i \lambda_i \ln \lambda_i)$ of the cosine kernel matrix.
   - **Within-Passage Half-to-Half Semantic Redundancy:** Cosine similarity between the first 600 chars and second 600 chars of each passage.
2. Check **empirical orthogonality** to surface lexical metrics (`distinct-2`, `rep-4`) across the 40 bundles per pair.

In [1]:
import json
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "analysis" / "out" / "critical_foundation_stats.json").exists():
    if (ROOT.parent / "analysis" / "out" / "critical_foundation_stats.json").exists():
        ROOT = ROOT.parent

data = json.loads((ROOT / "analysis" / "out" / "critical_foundation_stats.json").read_text(encoding="utf-8"))
p5 = data["phase5_semantic_diversity_on_600_real_passages"]

print("=== 1. CORPUS-LEVEL (100 PASSAGES/ARM) CONTEXTUAL & TOPICAL SEMANTIC DIVERSITY ===")
for pair_name, res in p5.items():
    c = res["corpus_100_passages"]
    o = res["orthogonality_to_lexical_metrics_40_bundles"]
    print(f"\n--- {pair_name} (100 Low-SC/High-gamma passages vs 100 High-SC/Low-gamma passages) ---")
    print(f"  T5-Small Raw Cosine Dist:        Low-SC = {c['t5_raw_cosine_dist_low_sc']:.4f} | High-SC = {c['t5_raw_cosine_dist_high_sc']:.4f} | Vendi: {c['t5_raw_vendi_score_low_sc']:.2f} vs {c['t5_raw_vendi_score_high_sc']:.2f}")
    print(f"  T5-Small Isotropic Vendi Score:  Low-SC = {c['t5_isotropic_vendi_score_low_sc']:.2f}   | High-SC = {c['t5_isotropic_vendi_score_high_sc']:.2f}")
    print(f"  GPT-2 Raw Cosine Dist (Aniso):   Low-SC = {c['gpt2_raw_cosine_dist_low_sc']:.5f}| High-SC = {c['gpt2_raw_cosine_dist_high_sc']:.5f}| Vendi: {c['gpt2_raw_vendi_score_low_sc']:.3f} vs {c['gpt2_raw_vendi_score_high_sc']:.3f}")
    print(f"  GPT-2 Isotropic Vendi Score:     Low-SC = {c['gpt2_isotropic_vendi_score_low_sc']:.2f}   | High-SC = {c['gpt2_isotropic_vendi_score_high_sc']:.2f}")
    print(f"  LSA Content-Topic Cosine Dist:   Low-SC = {c['lsa_topic_cosine_dist_low_sc']:.4f} | High-SC = {c['lsa_topic_cosine_dist_high_sc']:.4f} | Vendi: {c['lsa_topic_vendi_score_low_sc']:.2f} vs {c['lsa_topic_vendi_score_high_sc']:.2f}")
    print(f"  Orthogonality (40 bundles):      r(T5, distinct-2) = {o['r_t5_cos_vs_bundle_distinct2']:+.3f}, r(T5, rep-4) = {o['r_t5_cos_vs_bundle_rep4']:+.3f}, r(LSA, distinct-2) = {o['r_lsa_topic_vs_bundle_distinct2']:+.3f}")

=== 1. CORPUS-LEVEL (100 PASSAGES/ARM) CONTEXTUAL & TOPICAL SEMANTIC DIVERSITY ===

--- M1_ELFM (100 Low-SC/High-gamma passages vs 100 High-SC/Low-gamma passages) ---
  T5-Small Raw Cosine Dist:        Low-SC = 0.3092 | High-SC = 0.2992 | Vendi: 6.00 vs 5.86
  T5-Small Isotropic Vendi Score:  Low-SC = 59.43   | High-SC = 61.73
  GPT-2 Raw Cosine Dist (Aniso):   Low-SC = 0.00257| High-SC = 0.00316| Vendi: 1.024 vs 1.029
  GPT-2 Isotropic Vendi Score:     Low-SC = 20.78   | High-SC = 21.57
  LSA Content-Topic Cosine Dist:   Low-SC = 0.8828 | High-SC = 0.9017 | Vendi: 26.59 vs 27.76
  Orthogonality (40 bundles):      r(T5, distinct-2) = +0.524, r(T5, rep-4) = -0.166, r(LSA, distinct-2) = +0.566

--- S1_ELFB (100 Low-SC/High-gamma passages vs 100 High-SC/Low-gamma passages) ---
  T5-Small Raw Cosine Dist:        Low-SC = 0.3215 | High-SC = 0.3056 | Vendi: 6.21 vs 6.01
  T5-Small Isotropic Vendi Score:  Low-SC = 56.21   | High-SC = 59.60
  GPT-2 Raw Cosine Dist (Aniso):   Low-SC = 0.00242| 

In [1]:
print("=== 2. BUNDLE-LEVEL (20 BUNDLES vs 20 BUNDLES) WELCH t-TESTS & HEDGES' g ===")
print(f"{'Pair':<12} {'Semantic Metric':<30} {'Mean Low-SC':>12} {'Mean High-SC':>13} {'Diff (B-A)':>12} {'Hedges g':>10} {'p_Welch':>10}")
print("-" * 105)
for pair_name, res in p5.items():
    for m_name, st in res["bundle_level_welch_tests_20_bundles"].items():
        print(f"{pair_name:<12} {m_name:<30} {st['mean_A']:>12.4f} {st['mean_B']:>13.4f} {st['diff_B_minus_A']:>+12.4f} {st['hedges_g']:>+10.2f} {st['p_welch_2sided']:>10.4f}")
    wp = res["within_passage_half_to_half_semantic_redundancy_t5"]
    print(f"{pair_name:<12} {'within_half_to_half_sim_t5':<30} {wp['mean_A']:>12.4f} {wp['mean_B']:>13.4f} {wp['diff_B_minus_A']:>+12.4f} {wp['hedges_g']:>+10.2f} {wp['p_welch_2sided']:>10.4f}")

=== 2. BUNDLE-LEVEL (20 BUNDLES vs 20 BUNDLES) WELCH t-TESTS & HEDGES' g ===
Pair         Semantic Metric                 Mean Low-SC  Mean High-SC   Diff (B-A)   Hedges g    p_Welch
---------------------------------------------------------------------------------------------------------
M1_ELFM      t5_raw_cosine_dist                   0.3104        0.3028      -0.0076      -0.22     0.4831
M1_ELFM      t5_raw_vendi_score                   2.4366        2.4029      -0.0336      -0.20     0.5177
M1_ELFM      gpt2_raw_cosine_dist                 0.0026        0.0031      +0.0005      +0.55     0.0820
M1_ELFM      gpt2_isotropic_cosine_dist           1.0228        1.0113      -0.0116      -0.19     0.5534
M1_ELFM      lsa_topic_cosine_dist                0.8898        0.9043      +0.0145      +0.22     0.4893
M1_ELFM      within_half_to_half_sim_t5           0.8634        0.8673      +0.0040      +0.09     0.7529
S1_ELFB      t5_raw_cosine_dist                   0.3231        0.3079     